In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
font = {'size' : 15}
matplotlib.rc('font', **font)
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "Helvetica"
})
from skimage import measure
from scipy.spatial import ConvexHull
import os

In [ ]:
source_folder = "interface_data/"
os.makedirs("plots", exist_ok=True)

In [ ]:
fields = [0.13, 0.14, 0.15, 0.16]
fields_colors = matplotlib.colormaps.get('tab10')(range(4))

### FUNCTIONS

In [ ]:
def segment_noevents(data, no_event_val=-1):
    no_event_regions = [ r.astype('int') for r in measure.find_contours(data,no_event_val)]
    no_event_regions_sizes = np.array([r.shape[0] for r in no_event_regions])
    sorted_regions_idx = np.argsort(no_event_regions_sizes)
    inner_region = no_event_regions[sorted_regions_idx[-2]]
    outer_region = no_event_regions[sorted_regions_idx[-1]]
    center = inner_region.mean(axis=0)
    return inner_region,  outer_region, center

In [ ]:
def get_inner_mask(data, inner_region):
    # same as NB3: mask of the inner (initial) domain from the convex hull of inner_region
    nrows, ncols = data.shape
    ii, jj = np.meshgrid(np.arange(0,nrows), np.arange(0,ncols), indexing='ij')
    pts = np.vstack([ii.flatten(), jj.flatten()])
    inner_hull = ConvexHull(inner_region)
    inner_mask = np.ones((nrows, ncols), dtype=bool)
    for eq in inner_hull.equations:
        eq_mask = np.dot(eq[:-1], pts) + eq[-1] <= 1e-12
        inner_mask[ii.flatten(),jj.flatten()] *= eq_mask
    return inner_mask

In [ ]:
def shape_vs_time(data, inner_mask):
    # second moments of the region invaded up to t = inner domain + pixels with data <= t
    # computed for all t at once with cumulative sums over frames
    T = data.max()+1
    x_0, y_0 = np.argwhere(inner_mask*(data == -1)).T.astype('float')
    event_mask = data > -1
    frames = data[event_mask]
    x_ev, y_ev = np.argwhere(event_mask).T.astype('float')
    area_t = len(x_0) + np.cumsum(np.bincount(frames, minlength=T))
    def cumulative(w_0, w_ev):
        return (w_0.sum() + np.cumsum(np.bincount(frames, weights=w_ev, minlength=T)))/area_t
    m_x, m_y = cumulative(x_0, x_ev), cumulative(y_0, y_ev)
    c_xx = cumulative(x_0**2, x_ev**2) - m_x**2
    c_yy = cumulative(y_0**2, y_ev**2) - m_y**2
    c_xy = cumulative(x_0*y_0, x_ev*y_ev) - m_x*m_y
    tr = c_xx + c_yy
    det = c_xx*c_yy - c_xy**2
    l_plus = tr/2 + np.sqrt(tr**2/4 - det)
    l_min = tr/2 - np.sqrt(tr**2/4 - det)
    # direction of the major axis, same convention as theta in NB3: theta = arctan2(row - row_c, col - col_c)
    phi = 0.5*np.arctan2(2*c_xy, c_yy - c_xx)
    return area_t, l_plus, l_min, phi, m_x, m_y

In [ ]:
def fourier_modes(interface, theta_new, n_max):
    # R(theta,t) = c_0 [1 + sum_n a_n cos(n (theta - theta_n))]
    th = theta_new[:-1]   # theta_new[0] and theta_new[-1] are the same angle
    R = interface[:, :-1]
    c = np.array([ (R*np.exp(-1j*n*th)[None,:]).mean(axis=1) for n in range(0, n_max+1)]).T
    a_n = 2*np.abs(c[:, 1:])/np.abs(c[:, 0:1])
    theta_n = -np.angle(c[:, 1:])/np.arange(1, n_max+1)[None,:]
    return a_n, theta_n

In [ ]:
def coarse_grain(interface, theta_new, dth):
    # average R(theta,t) over windows of width dth
    th = theta_new[:-1]
    R = interface[:, :-1]
    w = max(1, int(round(dth/(th[1]-th[0]))))
    n_bins = len(th)//w
    R_cg = R[:, :n_bins*w].reshape(R.shape[0], n_bins, w).mean(axis=2)
    theta_cg = th[:n_bins*w].reshape(n_bins, w).mean(axis=1)
    return theta_cg, R_cg

In [ ]:
def block_velocity(R_cg, t_start, t_end, n_blocks):
    # mean velocity per theta bin in n_blocks consecutive time blocks of [t_start, t_end]
    t_edges = np.linspace(t_start, t_end, n_blocks+1).astype('int')
    v_blocks = np.array([ (R_cg[t_edges[b+1]]-R_cg[t_edges[b]])/(t_edges[b+1]-t_edges[b]) for b in range(n_blocks)])
    return v_blocks

### Shape factor of the invaded region vs $t$

In [ ]:
shape_dict = {}
for field in fields:
    print(field)
    data = np.loadtxt("data/file_field%1.2f.csv" % field, delimiter=',').astype('int')
    inner_region, outer_region, center = segment_noevents(data)
    inner_mask = get_inner_mask(data, inner_region)
    shape_dict[field] = shape_vs_time(data, inner_mask) + (center,)

In [ ]:
fig, ax = plt.subplots(figsize=(14,4), ncols=2)
for color, field in zip(fields_colors, fields):
    area_t, l_plus, l_min, phi, m_x, m_y, center = shape_dict[field]
    t = np.arange(len(area_t))
    ax[0].plot(t, np.sqrt(l_min/l_plus), color=color, label="$%1.2f \\, \\rm{mT}$" % field)
    ax[1].plot(t, np.degrees(phi), color=color)
ax[0].set_xlabel("$t$ (frames)")
ax[0].set_ylabel("$\\sqrt{\\lambda_-/\\lambda_+}$")
ax[0].legend()
ax[1].set_xlabel("$t$ (frames)")
ax[1].set_ylabel("major axis $\\phi$ (deg)")
ax[1].set_ylim(-90, 90)
plt.tight_layout()
plt.savefig("plots/shape_factor_t.pdf", bbox_inches='tight')
plt.show()

In [ ]:
# same as function of the equivalent radius R_eff = sqrt(A/pi), so fields with different speeds can be compared
fig, ax = plt.subplots(figsize=(7,4))
for color, field in zip(fields_colors, fields):
    area_t, l_plus, l_min, phi, m_x, m_y, center = shape_dict[field]
    ax.plot(np.sqrt(area_t/np.pi), np.sqrt(l_min/l_plus), color=color, label="$%1.2f \\, \\rm{mT}$" % field)
ax.set_xlabel("$R_{\\rm eff}=\\sqrt{A/\\pi}$ (px)")
ax.set_ylabel("$\\sqrt{\\lambda_-/\\lambda_+}$")
ax.legend()
plt.tight_layout()
plt.savefig("plots/shape_factor_Reff.pdf", bbox_inches='tight')
plt.show()

### Fourier modes of $R(\theta,t)$

In [ ]:
n_max = 4
modes_dict = {}
for field in fields:
    d = np.load("%sinterface_field=%1.2f.npz" % (source_folder,field))
    modes_dict[field] = fourier_modes(d['interface'], d['theta_new'], n_max)

In [ ]:
fig, ax = plt.subplots(figsize=(14,8), ncols=2, nrows=2)
ax = ax.flatten()
for i, (color, field) in enumerate(zip(fields_colors, fields)):
    a_n, theta_n = modes_dict[field]
    t = np.arange(a_n.shape[0])
    for n in range(1, n_max+1):
        ax[i].plot(t, a_n[:, n-1], label="$a_%i$" % n)
    ax[i].set_yscale('log')
    ax[i].set_title("$H=%1.2f$" % field)
    ax[i].set_xlabel("$t$ (frames)")
    ax[i].set_ylabel("$a_n$")
ax[0].legend()
plt.tight_layout()
plt.savefig("plots/fourier_modes_t.pdf", bbox_inches='tight')
plt.show()

### Velocity vs $\theta$, coarse grained over $\Delta\theta$

In [ ]:
t_start = 30000
n_blocks = 8
dths = [2*np.pi/64, 2*np.pi/32, 2*np.pi/16]

In [ ]:
fig, ax = plt.subplots(figsize=(14,8), ncols=2, nrows=2)
ax = ax.flatten()
for i, (color, field) in enumerate(zip(fields_colors, fields)):
    d = np.load("%sinterface_field=%1.2f.npz" % (source_folder,field))
    interface, theta_new = d['interface'], d['theta_new']
    t_end = interface.shape[0]-1
    for dth, ls in zip(dths, ['-', '--', ':']):
        theta_cg, R_cg = coarse_grain(interface, theta_new, dth)
        v_blocks = block_velocity(R_cg, t_start, t_end, n_blocks)
        v_mean = v_blocks.mean(axis=0)
        v_err = v_blocks.std(axis=0)/np.sqrt(n_blocks)
        ax[i].errorbar(np.degrees(theta_cg), v_mean/v_mean.mean(), yerr=v_err/v_mean.mean(), ls=ls, capsize=2, label="$\\Delta\\theta=2\\pi/%i$" % round(2*np.pi/dth))
    phi_final = np.degrees(shape_dict[field][3][t_start:].mean())
    for p in [phi_final, phi_final+180, phi_final-180]:
        if(-180 <= p <= 180):
            ax[i].axvline(p, color='grey', ls='-.')
    ax[i].axhline(1, color='black', lw=0.5)
    ax[i].set_xlim(-180, 180)
    ax[i].set_title("$H=%1.2f$, $t > %i$" % (field, t_start))
    ax[i].set_xlabel("$\\theta$ (deg)")
    ax[i].set_ylabel("$v(\\theta)/\\langle v \\rangle$")
ax[0].legend(fontsize=10)
plt.tight_layout()
plt.savefig("plots/velocity_theta.pdf", bbox_inches='tight')
plt.show()

In [ ]:
# polar view, one Delta theta, all fields
dth = 2*np.pi/32
fig = plt.figure(figsize=(6,6))
ax = fig.add_subplot(projection='polar')
for color, field in zip(fields_colors, fields):
    d = np.load("%sinterface_field=%1.2f.npz" % (source_folder,field))
    interface, theta_new = d['interface'], d['theta_new']
    theta_cg, R_cg = coarse_grain(interface, theta_new, dth)
    v_mean = block_velocity(R_cg, t_start, interface.shape[0]-1, n_blocks).mean(axis=0)
    ax.plot(np.append(theta_cg, theta_cg[0]), np.append(v_mean, v_mean[0])/v_mean.mean(), color=color, label="$%1.2f \\, \\rm{mT}$" % field)
ax.set_theta_direction(-1)   # theta grows towards increasing row index, i.e. downwards in the image
ax.set_ylim(0.6, 1.3)
ax.legend(loc='lower left', fontsize=10, bbox_to_anchor=(-0.1, -0.1))
plt.savefig("plots/velocity_theta_polar.pdf", bbox_inches='tight')
plt.show()

In [ ]:
# v(theta,t) map: is the anisotropy stable in time?
dth = 2*np.pi/16
dt = 5000
fig, ax = plt.subplots(figsize=(14,8), ncols=2, nrows=2)
ax = ax.flatten()
for i, field in enumerate(fields):
    d = np.load("%sinterface_field=%1.2f.npz" % (source_folder,field))
    interface, theta_new = d['interface'], d['theta_new']
    theta_cg, R_cg = coarse_grain(interface, theta_new, dth)
    n_win = (interface.shape[0]-1)//dt
    v_t = block_velocity(R_cg, 0, n_win*dt, n_win)
    im = ax[i].imshow((v_t/v_t.mean(axis=1)[:,None]).T, aspect='auto', origin='lower', cmap='seismic', vmin=0.7, vmax=1.3,
                      extent=[0, n_win*dt, np.degrees(theta_cg[0]), np.degrees(theta_cg[-1])])
    ax[i].set_title("$H=%1.2f$" % field)
    ax[i].set_xlabel("$t$ (frames)")
    ax[i].set_ylabel("$\\theta$ (deg)")
    plt.colorbar(im, ax=ax[i], label="$v(\\theta,t)/\\langle v \\rangle_\\theta$")
plt.tight_layout()
plt.savefig("plots/velocity_theta_t.pdf", bbox_inches='tight')
plt.show()

### $R(\theta,t)$ and $v(\theta,t)$ at multiple $\theta$

In [ ]:
# directions relative to the major axis phi of each field (stationary average)
dth = 2*np.pi/32
dt = 5000
dthetas_deg = [0, 45, 90, 135, 180]
thetas_colors = matplotlib.colormaps.get('viridis')(np.linspace(0, 0.9, len(dthetas_deg)))

In [ ]:
fig, ax = plt.subplots(figsize=(14,16), ncols=2, nrows=4)
for i, field in enumerate(fields):
    d = np.load("%sinterface_field=%1.2f.npz" % (source_folder,field))
    interface, theta_new = d['interface'], d['theta_new']
    theta_cg, R_cg = coarse_grain(interface, theta_new, dth)
    phi = shape_dict[field][3]
    phi_axis = 0.5*np.angle(np.mean(np.exp(2j*phi[t_start:])))   # average of an axis (defined mod pi)
    t = np.arange(R_cg.shape[0])
    n_win = (R_cg.shape[0]-1)//dt
    t_win = (np.arange(n_win)+0.5)*dt
    v_t = block_velocity(R_cg, 0, n_win*dt, n_win)
    for color, dtheta in zip(thetas_colors, dthetas_deg):
        theta_target = np.angle(np.exp(1j*(phi_axis + np.radians(dtheta))))
        k = np.argmin(np.abs(np.angle(np.exp(1j*(theta_cg - theta_target)))))
        lbl = "$\\phi+%i^\\circ$ ($\\theta=%i^\\circ$)" % (dtheta, round(np.degrees(theta_cg[k])))
        ax[i,0].plot(t, R_cg[:, k] - R_cg.mean(axis=1), color=color, label=lbl)
        ax[i,1].plot(t_win, v_t[:, k], color=color, marker='o', label=lbl)
    ax[i,1].plot(t_win, v_t.mean(axis=1), color='black', ls='dashed', label="$\\langle v \\rangle_\\theta$")
    ax[i,0].axhline(0, color='black', lw=0.5)
    ax[i,0].set_title("$H=%1.2f$" % field)
    ax[i,1].set_title("$H=%1.2f$, $\\Delta t=%i$" % (field, dt))
    ax[i,0].set_xlabel("$t$ (frames)")
    ax[i,1].set_xlabel("$t$ (frames)")
    ax[i,0].set_ylabel("$R(\\theta,t)-\\langle R \\rangle_\\theta$ (px)")
    ax[i,1].set_ylabel("$v(\\theta,t)$ (px/frame)")
    ax[i,0].legend(fontsize=9)
ax[0,1].legend(fontsize=9)
plt.tight_layout()
plt.savefig("plots/R_v_multiple_theta.pdf", bbox_inches='tight')
plt.show()

### Null tests: is the $n=2$ mode more than isotropic roughness?

In [ ]:
def mode_power_ratio(interface, t_list, n_fit_min, n_fit_max, n_test_max):
    # r_n = |c_n|^2 / (power law fitted on n_fit_min <= n < n_fit_max, extrapolated to n)
    # for an isotropic gaussian interface r_n is exponentially distributed with mean 1, so P(r > r_n) = exp(-r_n)
    R = interface[t_list, :-1]
    c = np.fft.rfft(R, axis=1)/R.shape[1]
    P = np.abs(c)**2
    n = np.arange(P.shape[1])
    bins = np.unique(np.logspace(np.log10(n_fit_min), np.log10(n_fit_max), 15).astype('int'))
    ratios = []
    for P_t in P:
        x = [ np.exp(np.mean(np.log(n[b0:b1]))) for b0, b1 in zip(bins[:-1], bins[1:])]
        y = [ P_t[b0:b1].mean() for b0, b1 in zip(bins[:-1], bins[1:])]
        slope, intercept = np.polyfit(np.log(x), np.log(y), 1)
        ratios.append(P_t[1:n_test_max+1]/(np.exp(intercept)*n[1:n_test_max+1]**slope))
    return np.array(ratios)

In [ ]:
# 1) amplitude: |c_n|^2 against the power law of the higher modes, along the stationary part
n_test_max = 6
fig, ax = plt.subplots(figsize=(14,8), ncols=2, nrows=2)
ax = ax.flatten()
for i, field in enumerate(fields):
    d = np.load("%sinterface_field=%1.2f.npz" % (source_folder,field))
    interface = d['interface']
    t_list = np.arange(t_start, interface.shape[0], 200)
    ratios = mode_power_ratio(interface, t_list, 7, 128, n_test_max)
    for n in range(1, n_test_max+1):
        ax[i].plot(t_list, ratios[:, n-1], lw=2 if n==2 else 1, color='black' if n==2 else None, label="$n=%i$ (median %.1f)" % (n, np.median(ratios[:, n-1])))
    ax[i].axhline(1, color='grey', lw=0.5)
    ax[i].axhline(-np.log(0.01), color='grey', ls='dashed', lw=1)   # p = 0.01 for a single time
    ax[i].set_yscale('log')
    ax[i].set_title("$H=%1.2f$" % field)
    ax[i].set_xlabel("$t$ (frames)")
    ax[i].set_ylabel("$r_n = |c_n|^2 / |c_n|^2_{\\rm fit}$")
    ax[i].legend(fontsize=9)
plt.tight_layout()
plt.savefig("plots/null_mode_power.pdf", bbox_inches='tight')
plt.show()

In [ ]:
# 2) phases: do the four fields agree on the direction of mode n?
# for 4 independent bubbles the phases n*theta_n are uniform -> p from random draws
n_max_phase = 8
phases = np.zeros((len(fields), n_max_phase))
for i, field in enumerate(fields):
    d = np.load("%sinterface_field=%1.2f.npz" % (source_folder,field))
    interface, theta_new = d['interface'], d['theta_new']
    t_list = np.arange(t_start, interface.shape[0], 200)
    R = interface[t_list, :-1]
    th = theta_new[:-1]
    for n in range(1, n_max_phase+1):
        phases[i, n-1] = -np.angle((R*np.exp(-1j*n*th)[None,:]).mean())   # = n*theta_n, from the time-averaged mode
R_bar = np.abs(np.exp(1j*phases).mean(axis=0))
random_R_bar = np.abs(np.exp(1j*np.random.uniform(0, 2*np.pi, (10**6, len(fields)))).mean(axis=1))
p_phase = np.array([ (random_R_bar >= rb).mean() for rb in R_bar])
for n in range(1, n_max_phase+1):
    print("n=%i  theta_n (deg): %s   R_bar=%.2f   p=%.4f" % (n, ' '.join('%5.0f' % np.degrees(np.angle(np.exp(1j*ph))/n) for ph in phases[:, n-1]), R_bar[n-1], p_phase[n-1]))
fig, ax = plt.subplots(figsize=(6,4))
ax.bar(np.arange(1, n_max_phase+1), p_phase, color=['black' if n==2 else 'grey' for n in range(1, n_max_phase+1)])
ax.axhline(0.05, color='red', ls='dashed')
ax.set_yscale('log')
ax.set_xlabel("$n$")
ax.set_ylabel("$p$ (phases aligned by chance)")
plt.tight_layout()
plt.savefig("plots/null_mode_phase.pdf", bbox_inches='tight')
plt.show()

In [ ]:
# 3) shared disorder? pixel-wise correlation of the arrival-time maps between fields (same image coordinates)
# tau = rank of the switching frame in [0,1], minus its average at the same distance from the center
# small scale = tau residual minus its gaussian blur (sigma_hp px); control = same with one map shifted by 60 px
from scipy.ndimage import gaussian_filter
sigma_hp = 10
H_crop, W_crop = 1000, 1040
eps_dict = {}
for field in fields:
    data = np.loadtxt("data/file_field%1.2f.csv" % field, delimiter=',').astype('int')
    inner_region, outer_region, center = segment_noevents(data)
    d = data[:H_crop, :W_crop]
    m = d > -1
    tau = np.zeros(d.shape)
    tau[m] = np.argsort(np.argsort(d[m]))/m.sum()
    ii, jj = np.indices(d.shape)
    r = np.hypot(ii-center[0], jj-center[1]).astype('int')
    tau_r = np.bincount(r[m], weights=tau[m], minlength=r.max()+1)/np.maximum(np.bincount(r[m], minlength=r.max()+1), 1)
    e = tau - tau_r[r]
    e[~m] = 0
    e_hp = e - gaussian_filter(e, sigma_hp)/np.maximum(gaussian_filter(m.astype('float'), sigma_hp), 1e-9)
    eps_dict[field] = (np.where(m, e, np.nan), np.where(m, e_hp, np.nan), m)
for a in range(len(fields)):
    for b in range(a+1, len(fields)):
        e_a, e_hp_a, m_a = eps_dict[fields[a]]
        e_b, e_hp_b, m_b = eps_dict[fields[b]]
        m = m_a*m_b
        e_hp_b_shift = np.roll(np.roll(e_hp_b, 60, 0), 60, 1)
        m_s = m*~np.isnan(e_hp_b_shift)
        print("%1.2f-%1.2f  large scale %.2f   small scale %.3f   small scale shifted %.3f" % (fields[a], fields[b],
              np.corrcoef(e_a[m], e_b[m])[0,1], np.corrcoef(e_hp_a[m], e_hp_b[m])[0,1], np.corrcoef(e_hp_a[m_s], e_hp_b_shift[m_s])[0,1]))